# 1단계 파싱

In [1]:
import json
from pathlib import Path
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
    ImageFormatOption
)
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions

from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline



# 모듈 최상단에 패턴 컴파일
import re
NEWLINE_PATTERN = re.compile(r'\r\n\d+')

def normalize_newlines(text: str) -> str:
    """개행문자 정규화 (동기 함수)"""
    return NEWLINE_PATTERN.sub('\n', text)

import os
from time import sleep, time
import pickle
import pdfplumber
from tqdm.auto import tqdm
from langchain_core.documents import Document


# 공유 가능한 옵션 정의
DEFAULT_PIPELINE_OPTIONS = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    ocr_options=EasyOcrOptions(lang=["en", "ko"])
    )


def parsing_pdf_by_page_with_docling(path:str, lv1_cat:str, lv2_cat:str, lv3_cat:str):
    path = path.replace("\\", "/")
    filename = path.split("/")[-1]

    pipeline_options = DEFAULT_PIPELINE_OPTIONS
    converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backend=PyPdfiumDocumentBackend
            ),}
    )
    loaded_docs = converter.convert(path)
    with pdfplumber.open(path) as pdf:
        page_num = 0
        docs = []
        for _ in tqdm(pdf.pages):
            docling_text = loaded_docs.document.export_to_markdown(page_no=int(page_num)+1)
            docling_text = docling_text.replace("<!-- image -->", "")
            docling_text = normalize_newlines(docling_text)
            lang_doc = Document(page_content=docling_text, metadata={'filename': filename, 'lv1_cat': lv1_cat, 'lv2_cat': lv2_cat, 'lv3_cat': lv3_cat, 'page':str(page_num)})
            docs.append(lang_doc)
            page_num+=1
            sleep(0.1)
    
    parsed_foldername = f"{lv1_cat}_{lv2_cat}"
    if not os.path.exists(f"./docs/{parsed_foldername}"):
        os.makedirs(f"./docs/{parsed_foldername}")
        
    parsed_filename = filename.replace(".pdf", "")
    with open(f"./docs/{parsed_foldername}/{parsed_filename}.pkl", 'ab') as file:
        pickle.dump(docs, file)

    # if os.path.exists(path):
    #     os.remove(path)

    return docs

d:\auto_vectordb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
start_time = time()
path = f"./file/KR Notation Guide_2025.pdf"
lv1_cat, lv2_cat, lv3_cat = "RULE", "KR", "NOTATION"
result = parsing_pdf_by_page_with_docling(path=path, lv1_cat=lv1_cat, lv2_cat=lv2_cat, lv3_cat=lv3_cat)
end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")

100%|██████████| 249/249 [02:32<00:00,  1.63it/s]

Document converted and tables exported in 7119.64 seconds.


In [3]:
with open("./docs/RULE_KR/KR Notation Guide_2025.pkl", 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [4]:
from IPython.display import Markdown
Markdown(loaded_object[2].page_content)

| 27-2. Dock Gate ············································································································ 179    |     |
|-------------------------------------------------------------------------------------------------------------------------------------|-----|
| 27-3. Launching Skid Barge  ·······················································································                 | 181 |
| 28. Refrigerated Cargo Carrier  ····················································································                | 183 |
| 29. Single Point Mooring  ·····························································································             | 185 |
| 30. Floating Structure  ···································································································         | 189 |
| 31. Shiplift and Transfer System  ···············································································                   | 192 |
| 32. WIG Craft ················································································································· 195 |     |
| 33. Floating LNG Bunkering Terminal  ·······································································                        | 199 |
| 34-1. Moored Oil Storage Tanker  ·············································································                      | 201 |
| 34-2. Moored Oil Storage Unit  ··················································································                   | 203 |
| 2-2 Remarks of SHIP TYPE  -  SPECIAL FEATURE NOTATIONS  ························ 205                                                |     |
| CHAPTER 3  ADDITIONAL SPECIAL FEATURE NOTATIONS ····································· 220                                           |     |
| CHAPTER 4  ADDITIONAL INSTALLATION NOTATIONS  ·········································· 231                                        |     |
| Annex 1 Written Examples of Class Notations  ·····························································                          | 235 |

# 페이지유형 구분(제목, 목차, 내용)

In [80]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b" , temperature=0)

from langchain.agents import create_agent
from langchain.messages import HumanMessage
system_prompt = """
You are an expert in analyzing Markdown document page structures.
Analyze the given Markdown text and classify the overall nature of the page into one of the following three categories:

1. Title
2. Table of Contents
3. Content

[Classification Definitions]

- Title:
  - A headline that represents the entire document or a single page
  - Even if multiple headers exist, classify as 'Title' if it serves as:
    ▶ a declaration of the document’s topic
    ▶ a year / version / rule name / guide name
    ▶ an introduction-like page
  - If there is no subordinate explanation or body text and it does not clearly serve a navigational purpose, treat it as a Title

- Table of Contents:
  - A structural guide page intended for internal document navigation
  - Classify as Table of Contents **only when the following characteristics are clearly present**:
    ▶ Section numbering systems (e.g., 1, 1.1, 2.3)
    ▶ Link or navigation expressions (#, [], page, jump, reference, etc.)
    ▶ Explicit labels such as "Contents" or "Table of Contents"
    ▶ Multiple items listed in a consistent format
  - A simple list of headers or year/introductory phrases does NOT qualify as a Table of Contents

- Content:
  - Contains concrete information such as explanatory sentences, rules, tables, data, procedures, or conditions, examples
  - If there is even a single explanatory sentence, classify it as Content

[Important Decision Rules]

- Multiple short headers ≠ Table of Contents
- Do NOT classify as Table of Contents if the navigation or linking purpose is not explicit
- If Title and Table of Contents are ambiguous, **always prioritize Title**

Output must be exactly one of the following, with no additional explanation:
'Title', 'Table of Contents', 'Content'
"""


agent = create_agent(
    llm,
    system_prompt=system_prompt,
)

def page_category(page_content):

    ## 첫줄 삭제    
    query = f"""Classify the below page content. ('Title', 'Table of Contents', 'Content')
    {page_content}
    """
    inputs = {"messages": [HumanMessage(content= query)]}
    res = agent.invoke(inputs)
    return res["messages"][-1].content

In [81]:
from tqdm import tqdm
re_docs = []
for doc in tqdm(loaded_object):
    page_type = page_category(doc.page_content)
    doc.metadata["page_type"] = page_type
    re_docs.append(doc)
len(re_docs)
re_docs[:3]

  2%|▏         | 4/249 [00:02<03:02,  1.35it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01hrm502bcfvcas4j2ppdwhsxf` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199821, Requested 1193. Please try again in 7m18.048s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
with open(f"./docs/RULE_KR/KR Notation Guide_2025_re.pkl", 'ab') as file:
        pickle.dump(re_docs, file)

In [75]:
with open("./docs/RULE_KR/KR Notation Guide_2025_re.pkl", 'rb') as file:
    loaded_object = pickle.load(file)

toc = []
for idx, d in enumerate(loaded_object):
    toc.append(f"{idx}_{d.metadata["page_type"]}")

toc

['0_Title',
 '1_Table of Contents',
 '2_Table of Contents',
 '3_Content',
 '4_Content',
 '5_Content',
 '6_Content',
 '7_Content',
 '8_Content',
 '9_Content',
 '10_Content',
 '11_Content',
 '12_Content',
 '13_Content',
 '14_Content',
 '15_Content',
 '16_Content',
 '17_Content',
 '18_Content',
 '19_Content',
 '20_Content',
 '21_Title',
 '22_Content',
 '23_Content',
 '24_Content',
 '25_Content',
 '26_Content',
 '27_Content',
 '28_Content',
 '29_Content',
 '30_Content',
 '31_Content',
 '32_Content',
 '33_Title',
 '34_Content',
 '35_Content',
 '36_Content',
 '37_Content',
 '38_Content',
 '39_Content',
 '40_Content',
 '41_Content',
 '42_Content',
 '43_Content',
 '44_Content',
 '45_Content',
 '46_Content',
 '47_Content',
 '48_Content',
 '49_Content',
 '50_Content',
 '51_Content',
 '52_Content',
 '53_Content',
 '54_Content',
 '55_Content',
 '56_Content',
 '57_Content',
 '58_Content',
 '59_Content',
 '60_Content',
 '61_Content',
 '62_Content',
 '63_Content',
 '64_Content',
 '65_Content',
 '66_C

In [79]:
Markdown(loaded_object[34].page_content)

## 3-2. NLS Tanker

## NOTATIONS (Ship Type Notations)

## NLS Tanker

## DESCRIPTIONS

NLS Tanker :

to be assigned to ships carrying only cargoes in bulk, except chemical(liquid cargoes specified in Pt 7, Ch 6, Sec 17 of the Rules), classified as pollution category Z, or category Z and OS, which are not subject to IBC Code and specified in Pt 7, Ch 6, Sec 18 of the Rules . (Noxious Liquid Substance)

## REQUIREMENTS / RULE REFERENCES

| Notations   | Design           | Survey    |
|-------------|------------------|-----------|
| NLS Tanker  | Pt 7 Ch 6 Sec 18 | Pt 1 Ch 2 |

## EXAMPLES

---------------------------------------------------------------------------------------------

✠KRS 1 - NLS Tanker

Category Z(18)

✠KRM 1

---------------------------------------------------------------------------------------------



# 목차 모아서 메타에 추가하기

In [38]:
with open("./docs/RULE_KR/KR Notation Guide_2025_re.pkl", 'rb') as file:
    loaded_object = pickle.load(file)

In [39]:
TOC = []
for d in tqdm(loaded_object):
    if d.metadata["page_type"] == "Table of Contents":
       TOC.append(d.page_content) 
len(TOC)

100%|██████████| 249/249 [00:00<00:00, 189818.56it/s]


2

In [40]:
toc = "\\n".join(TOC)
Markdown(toc)

## CONTENTS

| CHAPTER 1                                                                                                                             | GENERAL ···········································································································  1                  |
|---------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------|
| CHAPTER 2                                                                                                                             |                                                                                                                                         |
| 2-1 SHIP TYPE  -                                                                                                                      | SPECIAL FEATURE NOTATIONS ····················································· 5                                                       |
|                                                                                                                                       | 1. Oil Tanker ······················································································································· 5 |
| 2-1. Liquefied Gas Carrier                                                                                                            | ·····························································································  11                                       |
| 2-2. Compressed Natural Gas Carrier                                                                                                   | ········································································  20                                                            |
| 3-1. Chemical Tanker ····································································································· 24         |                                                                                                                                         |
| 3-2. NLS Tanker ·············································································································· 32     |                                                                                                                                         |
| 4. Oil/Chemical Tanker ··································································································· 35         |                                                                                                                                         |
| 5. Bulk Carrier ·················································································································· 45 |                                                                                                                                         |
| 6. Cargo Ship                                                                                                                         | ····················································································································  53                |
| 7. Ore Carrier ··················································································································· 58 |                                                                                                                                         |
| 8-1 Ore/Oil Carrier ·········································································································· 61     |                                                                                                                                         |
| 8-2 Ore/Chemical Carrier ······························································································ 66            |                                                                                                                                         |
| 8-3 Oil/Liquefied Gas Carrier  ·······················································································                | 74                                                                                                                                      |
| 9. Oil/Bulk/Ore Carrier ···································································································· 85       |                                                                                                                                         |
| 10. RoRo Ship                                                                                                                         | ··················································································································  92                  |
| 11. Container Ship                                                                                                                    | ···········································································································  96                         |
| 12. Fishing Vessel                                                                                                                    | ·········································································································  100                          |
| 13. Fish Carrier ·············································································································· 104   |                                                                                                                                         |
| 14. Passenger Ship                                                                                                                    | ·······································································································  107                            |
| 15-1. Tug Boat                                                                                                                        | ··············································································································  114                     |
| 15-2. Pusher ·················································································································· 118   |                                                                                                                                         |
| 16. Work Vessel  ············································································································         | 121                                                                                                                                     |
| 17. Special Purpose Ship                                                                                                              | ·····························································································  125                                      |
| 18. Barge                                                                                                                             | ···················································································································  130                |
| 19. Dredger                                                                                                                           | ···················································································································  137                |
| 20. Special Purpose Submersible  ···············································································                      | 140                                                                                                                                     |
| 21. Fixed Offshore Structure ······················································································ 146               |                                                                                                                                         |
| 22. Mobile Offshore Unit ····························································································· 150            |                                                                                                                                         |
| 23. Mobile Offshore Drilling Unit                                                                                                     | ···············································································  155                                                    |
| 24. Floating Production, Storage and Offloading Unit  ···········································                                     | 159                                                                                                                                     |
| 25-1. Floating LNG Storage and Regasification Unit  ············································                                      | 164                                                                                                                                     |
| 25-2. Floating LNG Production, Storage and Offloading Unit                                                                            | ·····························  168                                                                                                      |
| 26. Offshore Support Vessel  ······················································································                   | 172                                                                                                                                     |
| 27-1. Floating Dock                                                                                                                   | ······································································································  177                             |\n| 27-2. Dock Gate ············································································································ 179    |     |
|-------------------------------------------------------------------------------------------------------------------------------------|-----|
| 27-3. Launching Skid Barge  ·······················································································                 | 181 |
| 28. Refrigerated Cargo Carrier  ····················································································                | 183 |
| 29. Single Point Mooring  ·····························································································             | 185 |
| 30. Floating Structure  ···································································································         | 189 |
| 31. Shiplift and Transfer System  ···············································································                   | 192 |
| 32. WIG Craft ················································································································· 195 |     |
| 33. Floating LNG Bunkering Terminal  ·······································································                        | 199 |
| 34-1. Moored Oil Storage Tanker  ·············································································                      | 201 |
| 34-2. Moored Oil Storage Unit  ··················································································                   | 203 |
| 2-2 Remarks of SHIP TYPE  -  SPECIAL FEATURE NOTATIONS  ························ 205                                                |     |
| CHAPTER 3  ADDITIONAL SPECIAL FEATURE NOTATIONS ····································· 220                                           |     |
| CHAPTER 4  ADDITIONAL INSTALLATION NOTATIONS  ·········································· 231                                        |     |
| Annex 1 Written Examples of Class Notations  ·····························································                          | 235 |

In [41]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b" , temperature=0)

from langchain.agents import create_agent
from langchain.messages import HumanMessage
system_prompt = """
You are a document-structuring assistant.

Your task is to reorganize the rows and columns of the provided table of contents into a clean, structured tabular format.

Follow these rules strictly:

1. Hierarchy normalization
- Identify and preserve the hierarchical structure: Chapter → Section → Subsection.
- Normalize inconsistent numbering (e.g., 2-1, 3-2, 15-1) while preserving their original meaning.
- Do not invent or merge sections.

2. Row restructuring
- Each logical item must occupy exactly one row.
- Split compound lines into separate rows if they represent different entities.
- Remove visual separators (e.g., dotted leaders, dashed lines).

3. Column definition
- Output the table using the following columns, in this exact order:
  1) Chapter
  2) Section Number
  3) Title
  4) Category Type (e.g., Ship, Offshore Unit, Structure, Annex)
  5) Page

4. Content normalization
- Standardize capitalization for titles using Title Case.
- Trim extra whitespace and alignment characters.
- Keep original terminology; do not paraphrase technical names.

5. Special handling
- Treat Remarks, Annex, and Additional Notations as valid structural entries.
- Clearly associate them with their parent chapter.
- If a chapter heading has no page number, leave the Page column empty.

6. Output constraints
- Output only the reorganized table.
- Do not add explanations, comments, or summaries.
- Do not omit any entries from the original input.

The final output must be a clean, machine-readable table that accurately reflects the logical structure of the original contents.
"""


agent = create_agent(
    llm,
    system_prompt=system_prompt,
)

def reorganize_toc(toc):
    query = f""" reorganize the rows and columns of the below table of contents
    {toc}
    """
    inputs = {"messages": [HumanMessage(content= query)]}
    res = agent.invoke(inputs)
    return res["messages"][-1].content

In [42]:
re_toc = reorganize_toc(toc)
Markdown(re_toc)

| Chapter | Section Number | Title | Category Type | Page |
|---------|----------------|-------|---------------|------|
| 1 |  | General |  | 1 |
| 2 |  |  |  |  |
| 2 | 2-1 | Ship Type | Ship |  |
| 2 | 1 | Oil Tanker | Ship | 5 |
| 2 | 2-1 | Liquefied Gas Carrier | Ship | 11 |
| 2 | 2-2 | Compressed Natural Gas Carrier | Ship | 20 |
| 2 | 3-1 | Chemical Tanker | Ship | 24 |
| 2 | 3-2 | NLS Tanker | Ship | 32 |
| 2 | 4 | Oil/Chemical Tanker | Ship | 35 |
| 2 | 5 | Bulk Carrier | Ship | 45 |
| 2 | 6 | Cargo Ship | Ship | 53 |
| 2 | 7 | Ore Carrier | Ship | 58 |
| 2 | 8-1 | Ore/Oil Carrier | Ship | 61 |
| 2 | 8-2 | Ore/Chemical Carrier | Ship | 66 |
| 2 | 8-3 | Oil/Liquefied Gas Carrier | Ship | 74 |
| 2 | 9 | Oil/Bulk/Ore Carrier | Ship | 85 |
| 2 | 10 | RoRo Ship | Ship | 92 |
| 2 | 11 | Container Ship | Ship | 96 |
| 2 | 12 | Fishing Vessel | Ship | 100 |
| 2 | 13 | Fish Carrier | Ship | 104 |
| 2 | 14 | Passenger Ship | Ship | 107 |
| 2 | 15-1 | Tug Boat | Ship | 114 |
| 2 | 15-2 | Pusher | Ship | 118 |
| 2 | 16 | Work Vessel | Ship | 121 |
| 2 | 17 | Special Purpose Ship | Ship | 125 |
| 2 | 18 | Barge | Ship | 130 |
| 2 | 19 | Dredger | Ship | 137 |
| 2 | 20 | Special Purpose Submersible | Ship | 140 |
| 2 | 21 | Fixed Offshore Structure | Structure | 146 |
| 2 | 22 | Mobile Offshore Unit | Offshore Unit | 150 |
| 2 | 23 | Mobile Offshore Drilling Unit | Offshore Unit | 155 |
| 2 | 24 | Floating Production, Storage and Offloading Unit | Offshore Unit | 159 |
| 2 | 25-1 | Floating LNG Storage and Regasification Unit | Offshore Unit | 164 |
| 2 | 25-2 | Floating LNG Production, Storage and Offloading Unit | Offshore Unit | 168 |
| 2 | 26 | Offshore Support Vessel | Ship | 172 |
| 2 | 27-1 | Floating Dock | Structure | 177 |
| 2 | 27-2 | Dock Gate | Structure | 179 |
| 2 | 27-3 | Launching Skid Barge | Structure | 181 |
| 2 | 28 | Refrigerated Cargo Carrier | Ship | 183 |
| 2 | 29 | Single Point Mooring | Structure | 185 |
| 2 | 30 | Floating Structure | Structure | 189 |
| 2 | 31 | Shiplift and Transfer System | Structure | 192 |
| 2 | 32 | WIG Craft | Ship | 195 |
| 2 | 33 | Floating LNG Bunkering Terminal | Structure | 199 |
| 2 | 34-1 | Moored Oil Storage Tanker | Ship | 201 |
| 2 | 34-2 | Moored Oil Storage Unit | Structure | 203 |
| 2 | 2-2 | Remarks of Ship Type - Special Feature Notations | Ship | 205 |
| 3 |  | Additional Special Feature Notations |  | 220 |
| 4 |  | Additional Installation Notations |  | 231 |
| Annex 1 |  | Written Examples of Class Notations | Annex | 235 |

In [43]:
re_docs = []
for d in tqdm(loaded_object):
       d.metadata["table_of_contents"] = re_toc
       re_docs.append(d) 
len(re_docs)

100%|██████████| 249/249 [00:00<00:00, 183740.62it/s]


249

In [47]:
re_docs[0]

Document(metadata={'filename': 'KR Notation Guide_2025.pdf', 'lv1_cat': 'RULE', 'lv2_cat': 'KR', 'lv3_cat': 'NOTATION', 'page': '0', 'page_type': 'Title', 'table_of_contents': '| Chapter | Section Number | Title | Category Type | Page |\n|---------|----------------|-------|---------------|------|\n| 1 |  | General |  | 1 |\n| 2 |  |  |  |  |\n| 2 | 2-1 | Ship Type | Ship |  |\n| 2 | 1 | Oil Tanker | Ship | 5 |\n| 2 | 2-1 | Liquefied Gas Carrier | Ship | 11 |\n| 2 | 2-2 | Compressed Natural Gas Carrier | Ship | 20 |\n| 2 | 3-1 | Chemical Tanker | Ship | 24 |\n| 2 | 3-2 | NLS Tanker | Ship | 32 |\n| 2 | 4 | Oil/Chemical Tanker | Ship | 35 |\n| 2 | 5 | Bulk Carrier | Ship | 45 |\n| 2 | 6 | Cargo Ship | Ship | 53 |\n| 2 | 7 | Ore Carrier | Ship | 58 |\n| 2 | 8-1 | Ore/Oil Carrier | Ship | 61 |\n| 2 | 8-2 | Ore/Chemical Carrier | Ship | 66 |\n| 2 | 8-3 | Oil/Liquefied Gas Carrier | Ship | 74 |\n| 2 | 9 | Oil/Bulk/Ore Carrier | Ship | 85 |\n| 2 | 10 | RoRo Ship | Ship | 92 |\n| 2 | 11 | Cont

In [48]:
with open(f"./docs/RULE_KR/KR Notation Guide_2025_re2.pkl", 'ab') as file:
        pickle.dump(re_docs, file)

In [70]:
with open("./docs/RULE_KR/KR Notation Guide_2025_re2.pkl", 'rb') as file:
    loaded_object = pickle.load(file)
loaded_object[2:3]

[Document(metadata={'filename': 'KR Notation Guide_2025.pdf', 'lv1_cat': 'RULE', 'lv2_cat': 'KR', 'lv3_cat': 'NOTATION', 'page': '2', 'page_type': 'Table of Contents', 'table_of_contents': '| Chapter | Section Number | Title | Category Type | Page |\n|---------|----------------|-------|---------------|------|\n| 1 |  | General |  | 1 |\n| 2 |  |  |  |  |\n| 2 | 2-1 | Ship Type | Ship |  |\n| 2 | 1 | Oil Tanker | Ship | 5 |\n| 2 | 2-1 | Liquefied Gas Carrier | Ship | 11 |\n| 2 | 2-2 | Compressed Natural Gas Carrier | Ship | 20 |\n| 2 | 3-1 | Chemical Tanker | Ship | 24 |\n| 2 | 3-2 | NLS Tanker | Ship | 32 |\n| 2 | 4 | Oil/Chemical Tanker | Ship | 35 |\n| 2 | 5 | Bulk Carrier | Ship | 45 |\n| 2 | 6 | Cargo Ship | Ship | 53 |\n| 2 | 7 | Ore Carrier | Ship | 58 |\n| 2 | 8-1 | Ore/Oil Carrier | Ship | 61 |\n| 2 | 8-2 | Ore/Chemical Carrier | Ship | 66 |\n| 2 | 8-3 | Oil/Liquefied Gas Carrier | Ship | 74 |\n| 2 | 9 | Oil/Bulk/Ore Carrier | Ship | 85 |\n| 2 | 10 | RoRo Ship | Ship | 92 |\n| 

# 목차활용 글로벌 컨텍스트 추가 생성

In [ ]:
with open("./docs/RULE_KR/KR Notation Guide_2025_re2.pkl", 'rb') as file:
    loaded_object = pickle.load(file)

loaded_object[1]

Document(metadata={'filename': 'KR Notation Guide_2025.pdf', 'lv1_cat': 'RULE', 'lv2_cat': 'KR', 'lv3_cat': 'NOTATION', 'page': '1', 'page_type': 'Table of Contents', 'table_of_contents': '| Chapter | Section Number | Title | Category Type | Page |\n|---------|----------------|-------|---------------|------|\n| 1 |  | General |  | 1 |\n| 2 |  |  |  |  |\n| 2 | 2-1 | Ship Type | Ship |  |\n| 2 | 1 | Oil Tanker | Ship | 5 |\n| 2 | 2-1 | Liquefied Gas Carrier | Ship | 11 |\n| 2 | 2-2 | Compressed Natural Gas Carrier | Ship | 20 |\n| 2 | 3-1 | Chemical Tanker | Ship | 24 |\n| 2 | 3-2 | NLS Tanker | Ship | 32 |\n| 2 | 4 | Oil/Chemical Tanker | Ship | 35 |\n| 2 | 5 | Bulk Carrier | Ship | 45 |\n| 2 | 6 | Cargo Ship | Ship | 53 |\n| 2 | 7 | Ore Carrier | Ship | 58 |\n| 2 | 8-1 | Ore/Oil Carrier | Ship | 61 |\n| 2 | 8-2 | Ore/Chemical Carrier | Ship | 66 |\n| 2 | 8-3 | Oil/Liquefied Gas Carrier | Ship | 74 |\n| 2 | 9 | Oil/Bulk/Ore Carrier | Ship | 85 |\n| 2 | 10 | RoRo Ship | Ship | 92 |\n| 2

In [66]:
slm = ChatGroq(model="llama-3.3-70b-versatile" , temperature=0)

system_prompt = """
You are a document-level global context synthesis assistant.

Your task is to generate a GLOBAL CONTEXT SUMMARY for a specific page
within a structured technical document.

Inputs:
- The full Table of Contents (including chapter/section numbers and titles)
- The content of the current page
- The content of the immediately preceding page or section (if provided)

Your responsibilities:

1. Section identification
- Determine which Chapter and Section the current page belongs to.
- Use both section numbers AND section titles from the Table of Contents.
- Select the most specific section possible.

2. Fallback rule (critical):
- If the current page does NOT clearly map to a TOC section
  (e.g., missing section number, missing page number, or transitional content),
  infer the context based on the closest preceding section.
- Treat the preceding section as the authoritative structural anchor.

3. Context synthesis
- Generate a concise global context summary (5–10 lines).
- Explicitly mention:
  - Chapter number and title
  - Section/Subsection number and title (or the inferred preceding section)
- Explain how the current page relates to:
  - The preceding section
  - The overall purpose of this chapter
- Clarify whether the page introduces, extends, or elaborates on the section topic.

4. Constraints:
- Do NOT invent new sections.
- Do NOT explain your reasoning.
- Do NOT use bullet points, JSON, or headings.
- Do NOT quote large portions of the page verbatim.
- Write in neutral, technical documentation style.

Output format:
- Plain text only
- 5 to 10 lines
- This text will be used as a system-level global context for downstream agents.
"""

agent = create_agent(
    slm,
    system_prompt=system_prompt,
)


def generate_global_context(
    toc: str,
    current_page_content: str,
    previous_page_content: str | None = None
):
    query = f"""
[Table of Contents]
{toc}

[Current Page Content]
{current_page_content}
"""

    if previous_page_content:
        query += f"""

[Preceding Section or Page Content]
{previous_page_content}
"""

    query += """
Generate the global context summary for the current page.
"""

    inputs = {
        "messages": [HumanMessage(content=query)]
    }

    res = agent.invoke(inputs)
    return res["messages"][-1].content

In [69]:
redocs = []

for idx, d in tqdm(enumerate(loaded_object)):
    if d.metadata["page_type"] == "Content":
        
        if idx >= 1 and loaded_object[idx-1].metadata["page_type"] == "Content":
            previous_page_content = loaded_object[idx-1].page_content
        else: 
            previous_page_content = None

        current_page_content = d.page_content
        toc = d.metadata["table_of_contents"]
        global_context = generate_global_context(toc=toc, current_page_content=current_page_content, previous_page_content=previous_page_content)
        
        d.metadata["global_context"] = global_context

    else:
        print("Page Content가 아닙니다.")
        d.metadata["global_context"] = ""

    redocs.append(d)
    

0it [00:00, ?it/s]

Page Content가 아닙니다.
Page Content가 아닙니다.
Page Content가 아닙니다.


21it [01:28,  9.63s/it]

Page Content가 아닙니다.


33it [03:18, 10.41s/it]

Page Content가 아닙니다.


42it [04:41,  6.69s/it]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01hrm502bcfvcas4j2ppdwhsxf` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98618, Requested 2120. Please try again in 10m37.632s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
with open(f"./docs/RULE_KR/KR Notation Guide_2025_re3.pkl", 'ab') as file:
        pickle.dump(re_docs, file)

['Chapter 1, General, is the current context, with no specific section number or title provided. The preceding section is not applicable as this is the beginning of the document. This page introduces the general principles of ship classification and registration. It relates to the overall purpose of Chapter 1, which is to provide an overview of the classification process. The page elaborates on the assignment of class and registration of ships in the Register of Ships, setting the foundation for the subsequent chapters that delve into specific ship types and classification notations. The content of this page is foundational, providing a broad introduction to the topic.',
 'Chapter 2 of the document is focused on ship types and their respective notations. The current page belongs to Section 2, which encompasses various ship types and their classification. The preceding section, Chapter 1, provides a general overview of the classification of ships. The current page elaborates on the cons

In [ ]:
with open("./docs/RULE_KR/KR Notation Guide_2025_re3.pkl", 'rb') as file:
    loaded_object = pickle.load(file)
loaded_object[2:3]